# Лабораторная работа 8. Кластеризация и EM-алгоритм

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 7 |
| Опора на лекции | лекция 7: функционал качества кластеризации и метод $K$-средних (опр. 7.1), смеси распределений (опр. 7.4), EM-алгоритм (опр. 7.6) и его монотонность (теорема 7.7), ответственности (опр. 7.8), иерархическая кластеризация и linkage (опр. 7.11); лекция 6: байесовское правило, гауссовские плотности |
| Трудоёмкость | 2 ч аудиторно (части 1–3) + 4 ч самостоятельно |

## Цель работы

Реализовать $K$-средних и EM для смеси гауссиан, проверить монотонность правдоподобия из теоремы 7.7; понять, что $K$-средних — предельный случай EM, и увидеть, какие структуры данных он в принципе не способен найти; научиться выбирать число кластеров при отсутствии меток и осознать, что у задачи кластеризации нет единственного правильного ответа.

## Что нужно сдать

Заполненный ноутбук `lab08_student.ipynb`, в котором:

1. выполнены все задания (ячейки с `# TODO`), код запускается сверху вниз без ошибок;
2. под каждым заданием заполнены ячейки **Вывод** — своими словами, не пересказ кода;
3. в конце — раздел «Итоги работы» с ответами на контрольные вопросы;
4. все графики подписаны (заголовок, оси, легенда).

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже). Отчёт с чужим вариантом не принимается.

Кластеризация — задача **обучения без учителя**: ответов $y_i$ нет. Это меняет
всё, к чему мы привыкли в работах 1–7: нет ни эмпирического риска относительно
истинных ответов, ни скользящего контроля в прежнем смысле, ни однозначного
критерия «правильности». Поэтому здесь особенно важно понимать, **что именно**
оптимизирует каждый метод.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from scipy import stats
from sklearn.datasets import load_digits, load_iris, make_blobs, make_circles, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (adjusted_mutual_info_score, adjusted_rand_score,
                             silhouette_samples, silhouette_score)
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=8)
describe_variant(variant)

---
# Часть 1. $K$-средних

Определение 7.1: минимизируется сумма квадратов расстояний до центров кластеров

$$
Q(C, \mu) = \sum_{k=1}^{K}\sum_{i \in C_k}\|x_i - \mu_k\|^2 \;\longrightarrow\; \min .
$$

Алгоритм Ллойда чередует два шага, каждый из которых **не увеличивает** $Q$:

1. при фиксированных центрах отнести каждый объект к ближайшему центру;
2. при фиксированном разбиении пересчитать центры как средние кластеров.

Отсюда сходимость за конечное число шагов (значений $Q$ конечное число, и оно
не растёт), но **только к локальному минимуму**.

Реализуйте метод с двумя инициализациями: случайной и `k-means++`.

In [ ]:
def kmeans_pp_init(X, K, generator):
    """k-means++: следующий центр выбирается с вероятностью ~ квадрату
    расстояния до ближайшего из уже выбранных."""
    raise NotImplementedError


def kmeans(X, K, init="++", n_iter=200, tol=1e-10, seed=0):
    """Алгоритм Ллойда: чередование «отнести к ближайшему центру» и
    «пересчитать центры». Возвращает (метки, центры, история Q)."""
    raise NotImplementedError


X_b, y_b = make_blobs(n_samples=500, centers=4, cluster_std=1.1, random_state=RANDOM_STATE)
X_b = StandardScaler().fit_transform(X_b)

# TODO: 1) сверьте своё Q и разбиение со sklearn.cluster.KMeans (ARI ~ 1.0);
#       2) постройте график Q по итерациям -- он обязан убывать;
#       3) по 60 запусков с каждой инициализацией (случайная и k-means++):
#          гистограммы итогового Q и доля запусков, попавших в лучший минимум.

> **Вывод.** Убывает ли $Q$ монотонно? Какая инициализация чаще попадает в хороший локальный минимум и почему? Зачем в `sklearn` параметр `n_init`?
>
> *(ваш ответ здесь)*

---
# Часть 2. Что $K$-средних не умеет

Функционал $\sum_k\sum_{i \in C_k}\|x_i - \mu_k\|^2$ жёстко задаёт форму
кластеров: разбиение по ближайшему центру — это **диаграмма Вороного**, то есть
кластеры всегда выпуклые и разделены прямыми (гиперплоскостями). Кроме того,
квадрат расстояния штрафует крупные кластеры сильнее, чем мелкие.

Проверьте это на трудном случае вашего варианта: `variant["hard_case"]`.

In [ ]:
from sklearn.cluster import AgglomerativeClustering, DBSCAN, SpectralClustering

def make_hard(case, n=600, seed=RANDOM_STATE):
    """Сгенерировать трудный для K-средних случай из вашего варианта."""
    raise NotImplementedError


case = variant["hard_case"]

# TODO: постройте данные своего трудного случая и сравните на них пять методов:
#       K-средних, иерархическую с ward и single, DBSCAN, спектральную.
#       Нарисуйте разбиения и сведите в таблицу ARI, AMI и число найденных кластеров.

> **Вывод.** Насколько плохо сработал $K$-средних на вашем случае и **почему именно** — какое свойство функционала это объясняет? Какой метод справился и за счёт чего?
>
> *(ваш ответ здесь)*

---
# Часть 3. EM-алгоритм для смеси гауссиан

Определение 7.4: плотность смеси $p(x) = \sum_{k=1}^{K} w_k\, p_k(x)$,
$\sum_k w_k = 1$. Прямая максимизация правдоподобия по всем параметрам сложна
из-за логарифма суммы; EM (опр. 7.6) вводит скрытые переменные и чередует:

**E-шаг** — ответственности (опр. 7.8):
$$
\gamma_{ik} = \frac{w_k\, p_k(x_i)}{\sum_{s} w_s\, p_s(x_i)} ;
$$

**M-шаг** — пересчёт параметров как взвешенных оценок:
$$
w_k = \frac1\ell\sum_i \gamma_{ik}, \qquad
\mu_k = \frac{\sum_i \gamma_{ik} x_i}{\sum_i \gamma_{ik}}, \qquad
\Sigma_k = \frac{\sum_i \gamma_{ik}(x_i - \mu_k)(x_i - \mu_k)^{\mathsf T}}{\sum_i \gamma_{ik}} .
$$

Теорема 7.7: **логарифм правдоподобия не убывает** на каждой итерации.
Проверим это численно — это и есть главный эксперимент части.

Тип ковариации задаётся вашим вариантом: `variant["gmm_cov"]`.

In [ ]:
def gmm_em(X, K, cov_type="full", n_iter=200, tol=1e-8, seed=0, reg=1e-6):
    """EM для смеси гауссиан.

    E-шаг: gamma_ik = w_k p_k(x_i) / sum_s w_s p_s(x_i).
           Считайте в ЛОГАРИФМАХ (logpdf + трюк вычитания максимума),
           иначе при больших размерностях всё обнулится.
    M-шаг: w_k = mean(gamma_k); mu_k, Sigma_k -- взвешенные оценки.
           Для cov_type 'diag' оставьте только диагональ, для 'spherical' --
           единичную матрицу, умноженную на среднюю дисперсию.
    Возвращает (w, mu, Sigma, gamma, история logL).
    """
    raise NotImplementedError


cov_type = variant["gmm_cov"]
X_g, y_g = make_blobs(n_samples=600, centers=3, cluster_std=1.0, random_state=RANDOM_STATE)
X_g = X_g @ np.array([[0.6, -0.6], [-0.4, 0.8]])
X_g = StandardScaler().fit_transform(X_g)

# TODO: 1) обучите свой EM, выведите историю logL и ПРОВЕРЬТЕ, что все приросты
#          неотрицательны (теорема 7.7);
#       2) сверьтесь с sklearn.mixture.GaussianMixture по logL на объект и по ARI;
#       3) нарисуйте кластеры и эллипсы уровня ковариаций каждой компоненты;
#          прозрачностью точки покажите max_k gamma_ik -- уверенность отнесения.

### Задание 3.2. $K$-средних как предельный случай EM

Возьмём смесь со сферическими ковариациями $\Sigma_k = \sigma^2 I$ и устремим
$\sigma \to 0$. Ответственности
$\gamma_{ik} \propto w_k \exp\bigl(-\|x_i - \mu_k\|^2/(2\sigma^2)\bigr)$
превращаются в жёсткое отнесение к ближайшему центру, а M-шаг — в пересчёт
средних. То есть **$K$-средних — это EM для сферической смеси в пределе нулевой
дисперсии**.

Проверьте: посчитайте ответственности при убывающем $\sigma$ и сравните
разбиение с результатом $K$-средних.

In [ ]:
# TODO: возьмите центры, найденные K-средних, и для sigma из [1, 0.5, 0.2, 0.1, 0.05, 0.02]
#       посчитайте ответственности сферической смеси; проследите, как средняя
#       max-ответственность стремится к единице, а разбиение -- к разбиению K-средних.

> **Вывод.** Подтвердилась ли монотонность правдоподобия? Что происходит с ответственностями при $\sigma \to 0$? В каких ситуациях мягкое отнесение предпочтительнее жёсткого?
>
> *(ваш ответ здесь)*

---
# Часть 4. Выбор числа кластеров

Меток нет, поэтому «правильное» $K$ выбирается косвенно. Три подхода
(`variant["k_selection"]` указывает вашу пару):

* **метод локтя**: $Q(K)$ убывает всегда, ищут точку излома;
* **силуэт**: для объекта $i$
  $s_i = \dfrac{b_i - a_i}{\max(a_i, b_i)}$, где $a_i$ — среднее расстояние
  внутри своего кластера, $b_i$ — до ближайшего чужого; хорошо, когда
  средний силуэт велик;
* **информационные критерии** для смеси: $\mathrm{AIC} = -2\ln L + 2p$,
  $\mathrm{BIC} = -2\ln L + p\ln \ell$, где $p$ — число параметров модели.

Сравните их на данных, где истинное $K$ известно, — и посмотрите, совпадут ли ответы.

In [ ]:
X_k, y_k = make_blobs(n_samples=600, centers=4, cluster_std=1.15, random_state=RANDOM_STATE)
X_k = StandardScaler().fit_transform(X_k)
K_grid = range(2, 11)

# TODO: 1) для каждого K посчитайте Q (инерцию), средний силуэт, AIC и BIC
#          для GaussianMixture, а также ARI с истинными метками (для контроля);
#       2) выберите K каждым критерием (локоть -- максимум второй разности Q)
#          и сравните с истинным K = 4;
#       3) три графика: Q(K), силуэт(K), AIC и BIC(K);
#       4) силуэтная диаграмма для выбранного K.
print("критерии вашего варианта:", variant["k_selection"])

> **Вывод.** Совпали ли ответы разных критериев с истинным $K = 4$? Какой критерий оказался наименее надёжным? Почему $Q(K)$ нельзя минимизировать напрямую?
>
> *(ваш ответ здесь)*

---
# Часть 5. Иерархическая кластеризация

Определение 7.11: агломеративный алгоритм начинает с $\ell$ одноэлементных
кластеров и на каждом шаге объединяет два ближайших. Способ измерения расстояния
между кластерами (linkage) определяет всё поведение:

$$
\begin{aligned}
\text{single:}\ & R(U,V) = \min_{u \in U, v \in V}\rho(u,v), &
\text{complete:}\ & R(U,V) = \max_{u,v}\rho(u,v),\\
\text{average:}\ & R(U,V) = \tfrac{1}{|U||V|}\textstyle\sum_{u,v}\rho(u,v), &
\text{ward:}\ & \text{прирост } Q \text{ при слиянии}.
\end{aligned}
$$

Ваш вариант: `variant["linkage"]`. Постройте дендрограмму и сравните все четыре
способа.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage as scipy_linkage

# TODO: 1) для четырёх linkage постройте дендрограммы и соответствующие разбиения
#          на 3 кластера, посчитайте ARI;
#       2) воспроизведите «эффект цепочки»: две вытянутые полосы плюс десяток
#          точек-мостика между ними. Какие linkage склеивают полосы, а какие нет?
print("linkage вашего варианта:", variant["linkage"])

> **Вывод.** Как выглядят дендрограммы разных linkage и чем они отличаются? Какой linkage пострадал от эффекта цепочки и почему? Когда это свойство оказывается полезным?
>
> *(ваш ответ здесь)*

---
# Часть 6. Своя выборка

Кластеризуйте признаковое описание своей выборки (**без** использования целевой
переменной!) и проверьте, связаны ли найденные кластеры с целью. Это типичный
разведочный сценарий: кластеры могут отражать сегменты, которые полезно
рассматривать отдельно.

In [ ]:
data = load_personal(variant, return_frame=True)
# TODO: 1) кластеризуйте признаки (без целевой переменной!) для K = 2..8,
#          посчитайте Q, силуэт, BIC и AMI найденного разбиения с целью;
#       2) выберите K, постройте таблицу «кластер x класс» (доли по строкам)
#          и размеры кластеров;
#       3) выведите центры кластеров по 8 наиболее различающимся признакам
#          и объясните содержательно, чем кластеры отличаются.

## Итоги работы

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Почему $K$-средних всегда сходится, но не обязательно к глобальному минимуму? Приведите конфигурацию из четырёх точек и $K=2$, где неудачная инициализация даёт неоптимальный ответ.
2. Функционал $Q$ монотонно убывает по $K$. Означает ли это, что «чем больше кластеров, тем лучше»? Как правильно поставить вопрос о выборе $K$?
3. Теорема 7.7 гарантирует, что $\ln L$ не убывает. Гарантирует ли она сходимость к глобальному максимуму правдоподобия? Что делают на практике?
4. У вас данные из двух вытянутых, почти параллельных полос. Какой метод кластеризации вы возьмёте и почему $K$-средних не подойдёт?
5. Силуэт указал $K=2$, BIC — $K=5$. Ваши действия?

### Домашнее задание

1. **EM с неполными данными.** Обобщите свою реализацию EM на случай пропусков в признаках: на E-шаге ответственности считаются по наблюдаемым координатам (маргинальная плотность гауссианы — тоже гауссиана), на M-шаге пропущенные значения заменяются условным ожиданием $\mathbb{E}[x_{\text{miss}} \mid x_{\text{obs}}, k]$. Проверьте на своей выборке из работы 1 (там пропуски есть по построению): сравните качество восстановления пропущенных значений с `SimpleImputer(strategy='median')` по среднему квадрату ошибки на искусственно скрытых значениях.

2. **Устойчивость кластеризации как критерий выбора $K$.** Реализуйте подход, основанный на устойчивости: для каждого $K$ 50 раз возьмите случайные 80 % объектов, кластеризуйте и измерьте, насколько согласованы разбиения на пересечении подвыборок (ARI между парами). Постройте график средней согласованности от $K$ и сравните выбор с силуэтом и BIC на данных из части 4. Обоснуйте, почему устойчивость — разумный критерий именно для задачи без учителя.